<a href="https://colab.research.google.com/github/ReginaStrem/PIML-test-of-site-transferability/blob/main/PIML_descriptors_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PIML (P = f·g·h): сбор дескрипторов из открытых баз

Ноутбук дополняет базу `PIML_open_sources.sqlite` модулями, для которых нужен скрипт.
Каждый модуль независим: запускайте **Настройку** (разделы 0–1), затем нужные модули, в конце **Слияние** (раздел 15).
Результат каждого модуля сохраняется в `/content/piml/out/<модуль>.csv` (и в Google Drive, если он подключён).

| № | Модуль | Блок | Нужно от вас | Время |
|---|---|---|---|---|
| 2 | Элементные дескрипторы (mendeleev) | f, g | — | <1 мин |
| 3 | Materials Project | g | бесплатный API-ключ | 1–2 мин |
| 4 | JARVIS-DFT | g | — | 3–5 мин (скачивает ~1 ГБ в память) |
| 5 | Края зон по Батлеру–Джинли | g | проверить таблицу щелей | <1 мин |
| 6 | OCx24: DFT-дескрипторы | f | (опц.) большой файл 1,5 ГБ | 1 мин / 10–15 мин |
| 7 | Mok/Back DSTAR: энергии CO*, H*, OH* | f | — | 2–4 мин |
| 8 | SAC M@C₄₋ₓNₓ: сырые VASP | f | — | 2–3 мин |
| 9 | tmQM (M–N₄ комплексы) | g | — | 1–2 мин |
| 10 | Catalysis-Hub (Mamun) — опционально, у вас уже есть | f | — | 2–5 мин |
| 11 | NOMAD Schumann (хосты Cu, Ag) — экспериментальный | f | — | 5–15 мин |
| 12 | Атлас SAA/DAA (Berger) — таблицы из PDF SI | f | загрузить PDF | 1 мин |
| 13 | C2DB (2D-хосты) | g | загрузить c2db.db | 1 мин |
| 14 | Корпуса Sci. Data 2023/2024 | эксп. | загрузить файлы ScienceDB | 1 мин |
| 15 | Слияние в SQLite + Excel | — | загрузить PIML_open_sources.sqlite | <1 мин |

Модули 2, 5, 6 (малый файл), 7, 8, 9 проверены на реальных данных. Модули 3, 4, 10–14 написаны по документации баз; если что-то упадёт — пришлите текст ошибки.

## 0. Установка и настройки

In [3]:
!pip -q install mendeleev mp-api jarvis-tools pdfplumber ase pymatgen requests tqdm openpyxl

In [4]:
import os, re, io, json, gzip, math, time, sqlite3, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, requests
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

# ---------- что собираем (меняйте здесь) ----------
HOSTS_METAL = ["Au", "Ag", "Cu"]                       # плазмонные хосты SAA
DOPANTS = ["Fe","Co","Ni","Cu","Zn","Ru","Rh","Pd","Ag","Sn","Pt","Au","Bi",   # план синтеза
           "Ir","Os","Mn","Cr","V","Ti","Mo","W","Re","Ga","In","Cd"]          # DFT-ориентиры
SUPPORTS = ["SiO2","TiO2","ZnO","CeO2","WO3","SrTiO3","Cu2O","Fe2O3","BiVO4",
            "In2O3","SnO2","ZrO2","Al2O3","Nb2O5","Ta2O5","CdS","ZnS","MoS2","GaN","C3N4"]  # носители (хосты для g)
ALLOWED = sorted(set(HOSTS_METAL) | set(DOPANTS))

OUT = Path("/content/piml/out"); OUT.mkdir(parents=True, exist_ok=True)
RAW = Path("/content/piml/raw"); RAW.mkdir(parents=True, exist_ok=True)

# Google Drive (необязательно): копия результатов в MyDrive/PIML
USE_DRIVE = False
DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/PIML"); DRIVE_DIR.mkdir(parents=True, exist_ok=True)

def save(df, name, note=""):
    p = OUT / f"{name}.csv"; df.to_csv(p, index=False)
    if DRIVE_DIR: df.to_csv(DRIVE_DIR / f"{name}.csv", index=False)
    print(f"✓ {name}: {len(df)} строк, {df.shape[1]} столбцов -> {p}" + (f"  [{note}]" if note else ""))
    return p

def fetch(url, dest=None, chunk=1 << 20, timeout=120):
    # Скачивание с прогрессом. dest=None -> вернуть bytes.
    r = requests.get(url, stream=True, timeout=timeout); r.raise_for_status()
    total = int(r.headers.get("content-length", 0)) or None
    buf = open(dest, "wb") if dest else io.BytesIO()
    with tqdm(total=total, unit="B", unit_scale=True, desc=Path(url).name[:40], leave=False) as bar:
        for b in r.iter_content(chunk):
            buf.write(b); bar.update(len(b))
    if dest: buf.close(); return Path(dest)
    return buf.getvalue()

GH = "https://raw.githubusercontent.com"
print("Хосты:", HOSTS_METAL, "| допанты:", len(DOPANTS), "| носители:", len(SUPPORTS))

Хосты: ['Au', 'Ag', 'Cu'] | допанты: 25 | носители: 20


## 1. Вспомогательные функции для составов
`comp_dict("Ag-0.5-Au-0.5")`, разбор POSCAR и т. п.

In [5]:
from pymatgen.core import Composition

def comp_dict(s):
    # 'Ag-0.25-Au-0.75' -> {'Ag':0.25,'Au':0.75}
    t = s.split("-"); return {t[i]: float(t[i+1]) for i in range(0, len(t), 2)}

def host_and_minor(d):
    h = max(d, key=d.get); return h, ",".join(sorted(k for k in d if k != h))

def is_relevant(elements):
    # хотя бы один хост, все элементы из ALLOWED
    el = set(elements); return bool(el & set(HOSTS_METAL)) and el <= set(ALLOWED)

## 2. Элементные дескрипторы (mendeleev)
Для всех хостов, допантов и элементов носителей: ЭО Полинга, IE₁, EA, χ Малликена = (IE₁+EA)/2, радиусы, группа, число d-электронов.
`n_d` — число электронов во внешней d-оболочке по основной конфигурации (для Sn, Bi будет 10 — это заполненная оболочка).

In [6]:
from mendeleev import element

support_elements = sorted({e.symbol for f in SUPPORTS for e in Composition(f).elements})
EL_LIST = sorted(set(ALLOWED) | set(support_elements))

def n_d_outer(e):
    d = {k: v for k, v in e.ec.conf.items() if k[1] == "d"}
    return d[max(d)] if d else 0

rows = []
for s in EL_LIST:
    e = element(s)
    ie1 = e.ionenergies.get(1); ea = e.electron_affinity
    rows.append(dict(element=s, Z=e.atomic_number, group=e.group_id, period=e.period, block=e.block,
                     en_pauling=e.en_pauling, IE1_eV=ie1, EA_eV=ea,
                     chi_mulliken_eV=(ie1 + (ea if ea is not None and ea > 0 else 0.0)) / 2 if ie1 else None,
                     EA_missing_or_negative=(ea is None or ea <= 0),
                     atomic_radius_pm=e.atomic_radius, metallic_radius_pm=e.metallic_radius,
                     covalent_radius_pm=e.covalent_radius_pyykko, n_valence=e.nvalence(), n_d=n_d_outer(e),
                     role=("host" if s in HOSTS_METAL else "dopant" if s in DOPANTS else "support element")))
elements_df = pd.DataFrame(rows)
save(elements_df, "elements_mendeleev")
CHI = dict(zip(elements_df.element, elements_df.chi_mulliken_eV))
elements_df.head()

✓ elements_mendeleev: 36 строк, 16 столбцов -> /content/piml/out/elements_mendeleev.csv


,element,Z,group,period,block,en_pauling,IE1_eV,EA_eV,chi_mulliken_eV,EA_missing_or_negative,atomic_radius_pm,metallic_radius_pm,covalent_radius_pm,n_valence,n_d,role
0,Ag,47,11.0,5,d,1.93,7.576234,1.302000,4.439117,False,160.0,134.0,128.0,11,10,host
1,Al,13,13.0,3,p,1.61,5.985769,0.432830,3.209300,False,125.0,125.0,126.0,3,0,support element
2,Au,79,11.0,6,d,2.40,9.225554,2.308630,5.767092,False,135.0,134.0,124.0,11,10,host
3,Bi,83,15.0,6,p,1.90,7.285516,0.942362,4.113939,False,160.0,151.0,151.0,5,10,dopant
4,C,6,14.0,2,p,2.55,11.260288,1.262119,6.261203,False,70.0,NaN,75.0,4,0,support element


## 3. Materials Project (щели, края зон, стабильность)
Ключ бесплатный: <https://next-gen.materialsproject.org/api> → войти → скопировать API key.
В Colab удобнее сохранить его в **Secrets** (значок ключа слева) под именем `MP_API_KEY` — тогда вводить не нужно.
⚠️ Щели MP посчитаны в PBE и занижены в 1,5–2 раза; для краёв зон ниже приоритет у JARVIS-TBmBJ и экспериментальных щелей.

In [7]:
from mp_api.client import MPRester
try:
    from google.colab import userdata
    MP_API_KEY = userdata.get("MP_API_KEY")
except Exception:
    from getpass import getpass
    MP_API_KEY = getpass("Materials Project API key: ")

fields = ["material_id", "formula_pretty", "band_gap", "cbm", "vbm", "efermi", "is_gap_direct",
          "energy_above_hull", "formation_energy_per_atom", "density", "symmetry", "theoretical",
          "weighted_work_function"]
mp_rows = []
with MPRester(MP_API_KEY) as mpr:
    for f in tqdm(SUPPORTS):
        red = Composition(f).reduced_formula
        try:
            docs = mpr.materials.summary.search(formula=red, fields=fields)
        except Exception as ex:
            print("MP:", f, ex); continue
        for d in docs:
            mp_rows.append(dict(support=f, material_id=str(d.material_id), formula=d.formula_pretty,
                                spacegroup=d.symmetry.symbol if d.symmetry else None,
                                crystal_system=str(d.symmetry.crystal_system) if d.symmetry else None,
                                Eg_PBE_eV=d.band_gap, cbm_eV=d.cbm, vbm_eV=d.vbm, efermi_eV=d.efermi,
                                gap_direct=d.is_gap_direct, e_above_hull_eV=d.energy_above_hull,
                                dHf_eV_atom=d.formation_energy_per_atom, density=d.density,
                                theoretical=d.theoretical, work_function_eV=d.weighted_work_function))
mp_all = pd.DataFrame(mp_rows).sort_values(["support", "e_above_hull_eV"])
save(mp_all, "mp_supports_all_polymorphs")
mp_best = mp_all.groupby("support").head(1)          # самый стабильный полиморф
save(mp_best, "mp_supports_ground_state")
mp_best

Materials Project API key: ··········


  0%|          | 0/20 [00:00<?, ?it/s]

Output()

✓ Retrieved 322 SummaryDoc documents in 0.7s

Output()

✓ Retrieved 46 SummaryDoc documents in 0.3s

... (34 outputs left out)

Output()

✓ Retrieved 14 SummaryDoc documents in 0.2s

✓ mp_supports_all_polymorphs: 868 строк, 15 столбцов -> /content/piml/out/mp_supports_all_polymorphs.csv
✓ mp_supports_ground_state: 20 строк, 15 столбцов -> /content/piml/out/mp_supports_ground_state.csv


,support,material_id,formula,spacegroup,crystal_system,Eg_PBE_eV,cbm_eV,vbm_eV,efermi_eV,gap_direct,e_above_hull_eV,dHf_eV_atom,density,theoretical,work_function_eV
588,Al2O3,mp-1143,Al2O3,R-3c,Trigonal,5.8743,11.6968,5.8225,5.984054,True,0.000000,-3.426551,3.873499,False,None
459,BiVO4,mp-23044,VBiO4,I4_1/a,Tetragonal,2.2600,5.5621,3.3021,3.400218,False,0.000000,-2.154109,7.074339,False,None
855,C3N4,mp-971684,C3N4,Cmcm,Orthorhombic,1.1815,3.0216,1.8401,1.912232,False,0.088741,0.088741,1.943884,True,None
650,CdS,mp-672,CdS,P6_3mc,Hexagonal,1.1162,1.9763,0.8601,0.957524,True,0.000000,-0.776608,4.702285,False,None
383,CeO2,mp-20194,CeO2,Fm-3m,Cubic,2.0035,7.0786,5.0751,5.225255,False,0.000000,-3.927176,6.994759,False,None
430,Cu2O,mp-361,Cu2O,Pn-3m,Cubic,0.5127,4.6347,4.1220,4.177324,True,0.000000,-0.644505,6.204947,False,None
447,Fe2O3,mp-19770,Fe2O3,R-3c,Trigonal,0.0000,NaN,NaN,2.878285,False,0.000000,-1.707091,5.143372,False,None
851,GaN,mp-804,GaN,P6_3mc,Hexagonal,1.7376,5.1480,3.4104,3.594603,True,0.000000,-0.670049,6.080977,False,None
470,In2O3,mp-22598,In2O3,Ia-3,Cubic,0.9289,5.8419,4.9130,5.044626,False,0.000000,-1.994067,6.998712,False,None
807,MoS2,mp-1434,MoS2,R3m,Trigonal,1.3839,5.6577,4.2738,4.503485,False,0.000000,-0.966143,4.669645,False,None


## 4. JARVIS-DFT (щели OptB88vdW и TBmBJ, эффективные массы)
Набор `dft_3d` скачивается целиком (~1 ГБ в памяти), затем фильтруется по формулам носителей.

In [8]:
from jarvis.db.figshare import data as jdata
j = pd.DataFrame(jdata("dft_3d"))
print("JARVIS dft_3d:", j.shape)

def red(f):
    try: return Composition(f).reduced_formula
    except Exception: return None
targets = {Composition(f).reduced_formula: f for f in SUPPORTS}
j["reduced"] = j["formula"].map(red)
jj = j[j["reduced"].isin(targets)].copy()
jj["support"] = jj["reduced"].map(targets)
want = ["support", "jid", "formula", "spg_symbol", "optb88vdw_bandgap", "mbj_bandgap", "hse_gap",
        "ehull", "formation_energy_peratom", "avg_elec_mass", "avg_hole_mass", "epsx", "mepsx"]
jj = jj[[c for c in want if c in jj.columns]].replace("na", np.nan)
for c in jj.columns[3:]:
    jj[c] = pd.to_numeric(jj[c], errors="coerce")
jj = jj.sort_values(["support", "ehull"])
save(jj, "jarvis_supports_all_polymorphs")
jarvis_best = jj.groupby("support").head(1)
save(jarvis_best, "jarvis_supports_ground_state")
jarvis_best

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699


100%|██████████| 48.4M/48.4M [00:03<00:00, 15.3MiB/s]


Loading the zipfile...
Loading completed.
JARVIS dft_3d: (93902, 64)
✓ jarvis_supports_all_polymorphs: 315 строк, 13 столбцов -> /content/piml/out/jarvis_supports_all_polymorphs.csv
✓ jarvis_supports_ground_state: 20 строк, 13 столбцов -> /content/piml/out/jarvis_supports_ground_state.csv


,support,jid,formula,spg_symbol,optb88vdw_bandgap,mbj_bandgap,hse_gap,ehull,formation_energy_peratom,avg_elec_mass,avg_hole_mass,epsx,mepsx
1175,Al2O3,JVASP-32,Al2O3,NaN,6.430,7.571,NaN,1.5963,-3.31332,NaN,NaN,3.1744,2.7319
64997,BiVO4,JVASP-140556,VBiO4,NaN,2.131,NaN,NaN,2.3976,-2.14090,NaN,NaN,NaN,NaN
38754,C3N4,JVASP-30462,C3N4,NaN,1.089,NaN,NaN,6.3520,0.29215,NaN,NaN,7.3474,NaN
65771,CdS,JVASP-138544,CdS,NaN,0.971,2.353,NaN,0.3311,-0.66296,NaN,NaN,7.1489,4.6276
28884,CeO2,JVASP-1133,CeO2,NaN,2.211,NaN,NaN,0.8572,-3.68532,NaN,NaN,7.3102,NaN
20690,Cu2O,JVASP-1216,Cu2O,NaN,0.644,NaN,NaN,0.3903,-0.52398,1.010000,1.760000,8.5202,NaN
15635,Fe2O3,JVASP-99626,Fe2O3,NaN,0.409,0.734,NaN,2.7123,-1.44962,NaN,NaN,NaN,5.6400
35385,GaN,JVASP-30,GaN,NaN,1.943,3.084,NaN,1.2835,-0.57106,0.190000,1.646667,5.8255,4.4724
28038,In2O3,JVASP-12424,In2O3,NaN,1.195,2.786,NaN,0.9900,-1.70288,NaN,NaN,5.0409,NaN
23324,MoS2,JVASP-51,MoS2,NaN,0.955,1.293,NaN,2.5746,-0.96150,0.790000,0.820000,15.9665,15.1996


## 5. Края зон относительно NHE (Батлер–Джинли)
$E_{CB} = \chi - E^e - E_g/2$, $E_{VB} = E_{CB} + E_g$, где χ — среднее геометрическое χ Малликена атомов, $E^e = 4{,}5$ эВ.
Щель берётся по приоритету: **экспериментальная** (таблица ниже, проверьте и дополните) → JARVIS TBmBJ → MP PBE.
Значения `EXP_GAP` — типичные литературные щели при комнатной температуре; это стартовая точка, не догма: при наличии ваших UV-DRS ставьте свои.
Для полиморфов (анатаз/рутил) укажите нужный в `POLYMORPH` — иначе берётся самый стабильный.

In [9]:
E_E = 4.5
EXP_GAP = {   # эВ, типичные литературные значения — ПРОВЕРЬТЕ
    "TiO2": 3.2, "ZnO": 3.37, "CeO2": 3.0, "WO3": 2.7, "SrTiO3": 3.2, "Cu2O": 2.1, "Fe2O3": 2.1,
    "BiVO4": 2.4, "In2O3": 2.9, "SnO2": 3.6, "ZrO2": 5.0, "CdS": 2.4, "ZnS": 3.6, "MoS2": 1.3,
    "GaN": 3.4, "C3N4": 2.7, "SiO2": 8.9, "Al2O3": 8.8, "Nb2O5": 3.4, "Ta2O5": 4.0,
}

def chi_compound(formula):
    c = Composition(formula); n = c.num_atoms
    return math.exp(sum(amt * math.log(CHI[str(el)]) for el, amt in c.items()) / n)

def butler_ginley(chi, eg, e_e=E_E):
    ecb = chi - e_e - eg / 2; return ecb, ecb + eg

rows = []
for f in SUPPORTS:
    chi = chi_compound(f)
    mbj = jarvis_best.set_index("support").mbj_bandgap.get(f) if "jarvis_best" in globals() else None
    pbe = mp_best.set_index("support").Eg_PBE_eV.get(f) if "mp_best" in globals() else None
    for src, eg in [("exp", EXP_GAP.get(f)), ("TBmBJ", mbj), ("PBE", pbe)]:
        if eg is not None and not pd.isna(eg) and eg > 0:
            ecb, evb = butler_ginley(chi, eg)
            rows.append(dict(support=f, chi_eV=round(chi, 3), Eg_source=src, Eg_eV=eg,
                             E_CB_NHE_V=round(ecb, 3), E_VB_NHE_V=round(evb, 3)))
edges = pd.DataFrame(rows)
order = {"exp": 0, "TBmBJ": 1, "PBE": 2}
edges["rank"] = edges.Eg_source.map(order)
edges_best = edges.sort_values(["support", "rank"]).groupby("support").head(1).drop(columns="rank")
# CO2/CO2·− ≈ −1.9 V, CO2/CO −0.53 V, CO2/CH3OH −0.38 V, CO2/CH4 −0.24 V (pH 7, NHE) — для сравнения
edges_best["CB_above_CO2_CO"] = edges_best.E_CB_NHE_V < -0.53
# Батлер–Джинли плохо работает для g-C3N4 и слоистых халькогенидов (эксп. E_CB(g-C3N4) ≈ −1,1 В) — помечаем
edges_best["BG_unreliable"] = edges_best.support.isin(["C3N4", "MoS2"])
save(edges.drop(columns="rank"), "band_edges_all_gap_sources")
save(edges_best, "band_edges_best")
edges_best.sort_values("E_CB_NHE_V")

✓ band_edges_all_gap_sources: 51 строк, 6 столбцов -> /content/piml/out/band_edges_all_gap_sources.csv
✓ band_edges_best: 20 строк, 8 столбцов -> /content/piml/out/band_edges_best.csv


,support,chi_eV,Eg_source,Eg_eV,E_CB_NHE_V,E_VB_NHE_V,CB_above_CO2_CO,BG_unreliable
28,Al2O3,5.358,exp,8.80,-3.542,5.258,True,False
0,SiO2,6.473,exp,8.90,-2.477,6.423,True,False
46,GaN,4.833,exp,3.40,-1.367,2.033,True,False
26,ZrO2,5.855,exp,5.00,-1.145,3.855,True,False
40,ZnS,5.405,exp,3.60,-0.895,2.705,True,False
12,SrTiO3,5.317,exp,3.20,-0.783,2.417,True,False
20,In2O3,5.245,exp,2.90,-0.705,2.195,True,False
37,CdS,5.288,exp,2.40,-0.412,1.988,False,False
8,CeO2,5.603,exp,3.00,-0.397,2.603,False,False
3,TiO2,5.812,exp,3.20,-0.288,2.912,False,False


## 6. OCx24: DFT-дескрипторы адсорбции (C, H, OH, CO, CHO, COCOH)
**6a** — готовая таблица авторов: эксперимент + DFT-признаки (mean / Wulff / Boltzmann-усреднённые энергии) на каждый образец. 136 КБ, быстро.
**6b** — полный расчётный набор (19 406 материалов, 1,5 ГБ CSV) читается кусками и фильтруется по вашим элементам. Включите `RUN_OCX_BIG = True`.

In [10]:
base = f"{GH}/FAIR-Chem/fairchem/main/src/fairchem/applications/ocx/data"
proc = pd.read_csv(io.BytesIO(fetch(f"{base}/processed_data/CO2R_40_70_all.csv")))
proc = proc.loc[:, ~proc.columns.str.startswith("Unnamed")]
el_sets = proc["composition"].astype(str).map(lambda s: set(comp_dict(s)))
proc_rel = proc[el_sets.map(lambda e: bool(e & set(HOSTS_METAL)))].copy()
d = proc_rel["composition"].map(comp_dict)
proc_rel.insert(0, "host", d.map(lambda x: host_and_minor(x)[0]))
proc_rel.insert(1, "minor_elements", d.map(lambda x: host_and_minor(x)[1]))
save(proc_rel, "ocx24_processed_co2rr_AuAgCu", "эксперимент + DFT-признаки, все образцы с Au/Ag/Cu")
print([c for c in proc_rel.columns if c.endswith("_energy")])

CO2R_40_70_all.csv:   0%|          | 0.00/58.2k [00:00<?, ?B/s]

✓ ocx24_processed_co2rr_AuAgCu: 147 строк, 52 столбцов -> /content/piml/out/ocx24_processed_co2rr_AuAgCu.csv  [эксперимент + DFT-признаки, все образцы с Au/Ag/Cu]
['C_mean_energy', 'C_wulff_energy', 'C_boltz_energy', 'CO_mean_energy', 'CO_wulff_energy', 'CO_boltz_energy', 'CHO_mean_energy', 'CHO_wulff_energy', 'CHO_boltz_energy', 'COCOH_mean_energy', 'COCOH_wulff_energy', 'COCOH_boltz_energy', 'H_mean_energy', 'H_wulff_energy', 'H_boltz_energy', 'OH_mean_energy', 'OH_wulff_energy', 'OH_boltz_energy']


In [11]:
RUN_OCX_BIG = False   # True -> скачать 1,5 ГБ и отфильтровать
if RUN_OCX_BIG:
    url = "https://dl.fbaipublicfiles.com/opencatalystproject/data/ocx24/comp_df_241022.csv"
    big = RAW / "ocx24_comp_df_241022.csv"
    if not big.exists(): fetch(url, big)
    head = pd.read_csv(big, nrows=0).columns
    keep = [c for c in head if c in ("bulk_id", "slab_millers", "slab_shift", "top", "slab_comp", "n_bulk_elements",
                                      "cleavage_energy", "facet_fraction_on_wulff_not_normalized")
            or c.endswith("_min_sp_e") or c.endswith("_min_ml_Eads")]
    parts = []
    for ch in tqdm(pd.read_csv(big, usecols=keep, chunksize=200_000)):
        els = ch["slab_comp"].astype(str).map(lambda s: set(comp_dict(s)))
        parts.append(ch[els.map(is_relevant)])
    ocx_dft = pd.concat(parts, ignore_index=True)
    save(ocx_dft, "ocx24_dft_surfaces_relevant", "поверхности из ваших элементов с Au/Ag/Cu")
    agg_cols = [c for c in ocx_dft.columns if c.endswith("_min_sp_e")]
    ocx_comp = ocx_dft.groupby("slab_comp")[agg_cols].agg(["min", "median", "count"])
    ocx_comp.columns = ["_".join(c) for c in ocx_comp.columns]
    save(ocx_comp.reset_index(), "ocx24_dft_by_composition")

## 7. Mok/Back (DSTAR, Nat. Commun. 2023): DFT-энергии CO*, H*, OH* на сплавах
Файлы 17–67 МБ из GitHub. Из POSCAR берётся состав слэба (адсорбат — последние атомы), оставляются слэбы из ваших элементов с Au/Ag/Cu.
`energy` — энергия связывания адсорбата из GASpy (эВ, отсчёт как в статье).

In [12]:
DSTAR = f"{GH}/SeoinBack/DSTAR-CO2RR/HEAD/data"
ADS_N = {"CO": 2, "H": 1, "OH": 2}

def poscar_comp(p, n_ads):
    lines = p.splitlines()
    syms, cnts = lines[5].split(), [int(x) for x in lines[6].split()]
    atoms = [s for s, c in zip(syms, cnts) for _ in range(c)][:-n_ads]
    out = {}
    for a in atoms: out[a] = out.get(a, 0) + 1
    return out

mok_parts = []
for ads, n_ads in ADS_N.items():
    raw = fetch(f"{DSTAR}/{ads}_enumerated_data.csv")
    df = pd.read_csv(io.BytesIO(raw), usecols=["id", "poscar", "mpid", "miller", "shift", "top", "energy"])
    comp = df["poscar"].map(lambda p: poscar_comp(p, n_ads))
    df = df.drop(columns="poscar")
    df["slab_elements"] = comp.map(lambda c: "-".join(sorted(c)))
    df["slab_formula"] = comp.map(lambda c: Composition(c).reduced_formula)
    keep = comp.map(lambda c: is_relevant(c))
    df = df[keep].copy(); df["adsorbate"] = ads
    print(ads, ":", keep.sum(), "из", len(keep))
    mok_parts.append(df)
mok_dft = pd.concat(mok_parts, ignore_index=True).rename(columns={"energy": "E_bind_eV"})
save(mok_dft, "mok_dstar_dft_surfaces")
mok_agg = (mok_dft.groupby(["slab_formula", "slab_elements", "adsorbate"]).E_bind_eV
           .agg(["min", "median", "count"]).unstack("adsorbate"))
mok_agg.columns = [f"E_{a}_{s}_eV" if s != "count" else f"n_{a}" for s, a in mok_agg.columns]
save(mok_agg.reset_index(), "mok_dstar_by_composition")
mok_agg.head()

CO_enumerated_data.csv:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

CO : 36 из 18455


H_enumerated_data.csv:   0%|          | 0.00/12.4M [00:00<?, ?B/s]

H : 67 из 21317


OH_enumerated_data.csv:   0%|          | 0.00/5.31M [00:00<?, ?B/s]

OH : 45 из 6512
✓ mok_dstar_dft_surfaces: 148 строк, 9 столбцов -> /content/piml/out/mok_dstar_dft_surfaces.csv
✓ mok_dstar_by_composition: 28 строк, 11 столбцов -> /content/piml/out/mok_dstar_by_composition.csv


,,E_CO_min_eV,E_H_min_eV,E_OH_min_eV,E_CO_median_eV,E_H_median_eV,E_OH_median_eV,n_CO,n_H,n_OH
slab_formula,slab_elements,,,,,,,,,
Ag3Pd,Ag-Pd,0.355464,0.191681,0.872880,0.355464,0.191681,0.873657,1.0,1.0,2.0
Ag3Sn,Ag-Sn,0.098119,0.479987,0.310924,0.098119,0.591885,0.622198,1.0,3.0,3.0
AgAu3,Ag-Au,-0.045791,0.149354,1.070636,0.032395,0.199511,1.153121,3.0,4.0,3.0
Cu3Pd,Cu-Pd,-0.814642,-0.217224,0.264311,-0.814642,-0.217224,0.264311,1.0,1.0,1.0
Cu5Sn4,Cu-Sn,NaN,1.235369,NaN,NaN,1.235369,NaN,NaN,1.0,NaN


## 8. SAC M@C₄₋ₓNₓ (He, Wang, Ouyang 2023): энергии из сырых VASP
В репозитории нет готовой таблицы — только расчёты (700 OSZICAR). Берём последнюю E0 и считаем ΔE = E(X*) − E(*).
⚠️ Для части металлов (Co, Cr, Fe, Mn, Mo, Ni, Os в 0N0) энергия подложки рассогласована с расчётами адсорбатов (ΔE_H ≈ −12 эВ вместо ≈ −2) — такие строки помечаются `flag_inconsistent` и **не должны** идти в модель без ручной проверки.
ΔG не считается: нужны газовые референсы и ZPE/TS-поправки из той же постановки, их в репозитории нет.

In [13]:
api = requests.get("https://api.github.com/repos/jeff-oakley/SAC_HOER_data/git/trees/HEAD?recursive=1", timeout=60).json()
if "tree" not in api:
    raise RuntimeError(f"GitHub API: {api.get('message')} — подождите час (лимит 60 запросов/ч) или запустите позже")
osz = [t["path"] for t in api["tree"] if t["path"].endswith("OSZICAR")]
print("OSZICAR:", len(osz))

def last_e0(path):
    try:
        t = requests.get(f"{GH}/jeff-oakley/SAC_HOER_data/HEAD/{path}", timeout=60).text
        m = re.findall(r"E0=\s*([-\d.E+]+)", t); return path.split("/")[1], float(m[-1]) if m else None
    except Exception:
        return path.split("/")[1], None

with ThreadPoolExecutor(16) as ex:
    E = dict(tqdm(ex.map(last_e0, osz), total=len(osz)))

rows = []
for key in sorted({k.rsplit("_", 1)[0] for k in E if k.endswith("_substrate")}):
    coord, metal = key.split("_", 1)
    s = E.get(f"{key}_substrate"); r = dict(site_code=coord, metal=metal, E_substrate_eV=s)
    for a in ["H", "O", "OH", "OOH"]:
        e = E.get(f"{key}_{a}"); r[f"dE_{a}_raw_eV"] = (e - s) if (e is not None and s is not None) else None
    rows.append(r)
sac = pd.DataFrame(rows)
med = sac.groupby("site_code").dE_H_raw_eV.transform("median")
sac["flag_inconsistent"] = (sac.dE_H_raw_eV - med).abs() > 5
sac["in_DOPANTS"] = sac.metal.isin(DOPANTS)
save(sac, "sac_MC4xNx_raw_energies", f"{int(sac.flag_inconsistent.sum())} строк с флагом рассогласования")
sac[sac.flag_inconsistent].head(10)

OSZICAR: 700


  0%|          | 0/700 [00:00<?, ?it/s]

✓ sac_MC4xNx_raw_energies: 140 строк, 9 столбцов -> /content/piml/out/sac_MC4xNx_raw_energies.csv  [41 строк с флагом рассогласования]


,site_code,metal,E_substrate_eV,dE_H_raw_eV,dE_O_raw_eV,dE_OH_raw_eV,dE_OOH_raw_eV,flag_inconsistent,in_DOPANTS
3,0N0,Co,-1160.5456,-12.7533,-15.2031,-19.9731,-24.3137,True,True
4,0N0,Cr,-1162.5369,-13.1204,-17.6194,-21.5951,-25.7778,True,True
6,0N0,Fe,-1160.7493,-13.4668,-17.2642,-21.0791,-25.3220,True,True
10,0N0,Mn,-1162.7468,-12.4669,-16.7512,-20.6901,-24.9602,True,True
11,0N0,Mo,-1162.1375,-14.9611,-19.8103,-23.4383,-27.6052,True,True
13,0N0,Ni,-1157.8369,-14.0477,-15.2840,-20.6138,-25.0156,True,True
14,0N0,Os,-1166.7195,-10.7587,-15.4112,-18.8051,-22.9441,True,True
23,0N0,V,-1162.3877,-13.2249,-17.5404,-21.8853,-26.0835,True,True
24,0N0,W,-1161.6563,-17.5549,-22.5486,-26.0325,-30.1304,True,True
31,1N1,Co,-1160.5723,-13.4092,-15.4767,-20.0843,-24.7244,True,True


## 9. tmQM: молекулярные комплексы металлов (HOMO/LUMO, щель, заряд металла)
108 тыс. комплексов из CSD. Энергии в tmQM — в Хартри, переводим в эВ. Металл и число атомов N берём из `tmQM_X.q`.
Это прокси для «молекулярного» края зон одиночного центра M–Nₓ; к SAA на металлах относится косвенно.

In [14]:
TMQM = f"{GH}/uiocompcat/tmQM/master/tmQM"
y = pd.read_csv(io.BytesIO(fetch(f"{TMQM}/tmQM_y.csv")), sep=";")
qtxt = fetch(f"{TMQM}/tmQM_X.q").decode()
TM = {"Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn","Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
      "La","Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg"}
meta = []
for block in qtxt.split("CSD_code = ")[1:]:
    lines = block.strip().splitlines()
    code_ = lines[0].split("|")[0].strip()
    atoms = [l.split()[0] for l in lines[1:] if l.strip() and not l.startswith("Total")]
    metals = [a for a in atoms if a in TM]
    meta.append(dict(CSD_code=code_, metal=metals[0] if metals else None, n_metal=len(metals),
                     n_N=atoms.count("N"), n_atoms=len(atoms)))
meta = pd.DataFrame(meta)
H2EV = 27.211386
t = y.merge(meta, on="CSD_code", how="left")
for c in ["HL_Gap", "HOMO_Energy", "LUMO_Energy"]:
    t[c + "_eV"] = t[c] * H2EV
t = t[t.metal.isin(DOPANTS) & (t.n_metal == 1)]
save(t.drop(columns=["SMILES"], errors="ignore"), "tmqm_single_metal_complexes")
tm_agg = t.groupby("metal")[["HOMO_Energy_eV", "LUMO_Energy_eV", "HL_Gap_eV", "Metal_q"]].median()
tm_agg["n_complexes"] = t.groupby("metal").size()
tmN4 = t[t.n_N >= 4].groupby("metal")[["HOMO_Energy_eV", "LUMO_Energy_eV", "HL_Gap_eV", "Metal_q"]].median().add_suffix("_N4plus")
save(tm_agg.join(tmN4).reset_index(), "tmqm_by_metal")
tm_agg.join(tmN4)

tmQM_y.csv:   0%|          | 0.00/6.99M [00:00<?, ?B/s]

tmQM_X.q:   0%|          | 0.00/24.3M [00:00<?, ?B/s]

✓ tmqm_single_metal_complexes: 100396 строк, 17 столбцов -> /content/piml/out/tmqm_single_metal_complexes.csv
✓ tmqm_by_metal: 21 строк, 10 столбцов -> /content/piml/out/tmqm_by_metal.csv


,HOMO_Energy_eV,LUMO_Energy_eV,HL_Gap_eV,Metal_q,n_complexes,HOMO_Energy_eV_N4plus,LUMO_Energy_eV_N4plus,HL_Gap_eV_N4plus,Metal_q_N4plus
metal,,,,,,,,,
Ag,-5.708132,-2.699642,3.610951,0.468080,1801,-7.814566,-3.779117,3.698572,0.566420
Au,-5.687452,-2.076637,3.542242,0.211995,4338,-7.707625,-3.816533,3.402512,0.274860
Cd,-5.426903,-2.265348,3.256931,1.195290,1802,-5.396834,-2.277049,3.202780,1.216870
Co,-5.142680,-2.558142,2.700458,0.306795,3930,-5.861196,-3.755852,2.683995,0.418080
Cr,-5.105264,-1.969968,3.084138,-1.551685,2102,-4.166199,-2.399908,1.851599,0.015770
Cu,-4.898322,-2.418276,2.806038,0.428030,3816,-6.383791,-3.518704,2.579095,0.524830
Fe,-4.617772,-2.083576,2.598687,-0.195270,5691,-4.322393,-2.236096,2.167251,0.342550
Ir,-4.948391,-2.000717,3.062778,0.047200,5232,-5.122543,-2.393241,2.837875,0.315100
Mn,-5.274927,-2.633518,2.667532,-0.775900,2281,-6.573455,-4.176948,1.606288,0.583540


## 10. Catalysis-Hub: набор Mamun 2019 (опционально — у вас уже есть `mamun.py`)
GraphQL-API, постранично. Оставляются поверхности с Au/Ag/Cu.

In [15]:
RUN_CATHUB = False
if RUN_CATHUB:
    API = "https://api.catalysis-hub.org/graphql"
    q = '''{reactions(first: 500, pubId: "MamunHighT2019"%s) {
      totalCount pageInfo { hasNextPage endCursor }
      edges { node { Equation reactionEnergy surfaceComposition facet sites reactants products } } } }'''
    rows, after = [], ""
    with tqdm() as bar:
        while True:
            r = requests.post(API, json={"query": q % after}, timeout=120).json()["data"]["reactions"]
            rows += [e["node"] for e in r["edges"]]; bar.total = r["totalCount"]; bar.update(len(r["edges"]))
            if not r["pageInfo"]["hasNextPage"]: break
            after = f', after: "{r["pageInfo"]["endCursor"]}"'
    ch = pd.DataFrame(rows)
    els = ch.surfaceComposition.map(lambda s: set(re.findall(r"[A-Z][a-z]?", str(s))))
    ch = ch[els.map(lambda e: bool(e & set(HOSTS_METAL)))]
    save(ch, "cathub_mamun_AuAgCu")

## 11. NOMAD: Schumann et al. 2024 — хосты Cu и Ag (экспериментальный модуль)
В Source Data статьи есть только хост Au (уже в базе). Здесь вытаскиваются все записи набора из NOMAD с полной энергией и путём файла;
пары «слэб / слэб+адсорбат» надо сопоставить по пути (`mainfile`) — посмотрите структуру папок в выводе и допишите правило.
Энергия в архиве NOMAD хранится в джоулях — переводится в эВ.

In [16]:
RUN_NOMAD = False
if RUN_NOMAD:
    NOMAD = "https://nomad-lab.eu/prod/v1/api/v1"
    DOI = "10.17172/NOMAD/2023.12.04-2"
    ds = requests.get(f"{NOMAD}/datasets/", params={"doi": DOI}, timeout=60).json()
    ds_id = ds["data"][0]["dataset_id"]; print("dataset_id:", ds_id)
    entries, after = [], None
    while True:
        body = {"owner": "public", "query": {"datasets.dataset_id": ds_id},
                "pagination": {"page_size": 200, **({"page_after_value": after} if after else {})},
                "required": {"include": ["entry_id", "mainfile", "results.material.chemical_formula_reduced"]}}
        r = requests.post(f"{NOMAD}/entries/query", json=body, timeout=120).json()
        entries += r["data"]; after = r["pagination"].get("next_page_after_value")
        if not after: break
    print("записей:", len(entries))
    MAX_ENTRIES = 3000
    def total_energy(eid):
        try:
            a = requests.get(f"{NOMAD}/entries/{eid}/archive", timeout=120).json()["data"]["archive"]
            return a["run"][0]["calculation"][-1]["energy"]["total"]["value"] / 1.602176634e-19
        except Exception:
            return None
    ids = [e["entry_id"] for e in entries[:MAX_ENTRIES]]
    with ThreadPoolExecutor(8) as ex:
        en = list(tqdm(ex.map(total_energy, ids), total=len(ids)))
    nom = pd.DataFrame([{"entry_id": e["entry_id"], "mainfile": e.get("mainfile"),
                         "formula": e.get("results", {}).get("material", {}).get("chemical_formula_reduced")}
                        for e in entries[:MAX_ENTRIES]])
    nom["E_total_eV"] = en
    save(nom, "nomad_schumann_raw", "нужно сопоставить слэб/адсорбат по mainfile")
    print(nom.mainfile.head(20).tolist())

## 12. Атлас SAA/DAA (Berger, Wang, Sykes, Michaelides): таблицы S9–S39 из PDF SI
Поставьте `RUN_BERGER = True` и загрузите PDF Supporting Information. Все таблицы сохраняются в один длинный CSV с номером страницы и ближайшей подписью «Table S…» (если подпись стоит под таблицей, номер может сдвинуться на один — сверьте по странице). Таблицы без линеек разбираются по выравниванию текста; полный текст страниц сохраняется отдельно как запасной вариант.
Проверьте номер arXiv/журнала: в нашем списке стоит arXiv:2609.19087, его я не сверял.

In [17]:
RUN_BERGER = False
if RUN_BERGER:
  import pdfplumber
  from google.colab import files
  up = files.upload()                       # выберите PDF SI атласа
  pdf_path = next(iter(up))
  out, texts = [], []
  with pdfplumber.open(io.BytesIO(up[pdf_path])) as pdf:
      for pno, page in enumerate(tqdm(pdf.pages), 1):
          txt = page.extract_text() or ""
          caps = re.findall(r"Table\s*S\s?(\d+)", txt)
          tbls = page.extract_tables()
          if not tbls and caps:   # таблицы без линеек: разбор по выравниванию текста
              tbls = page.extract_tables({"vertical_strategy": "text", "horizontal_strategy": "text"})
          texts.append(dict(page=pno, captions=",".join("S" + c for c in caps), text=txt))
          for ti, tbl in enumerate(tbls):
              df = pd.DataFrame(tbl)
              df.insert(0, "table_id", f"S{caps[ti]}" if ti < len(caps) else (f"S{caps[-1]}" if caps else "?"))
              df.insert(0, "page", pno)
              out.append(df)
  berger = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
  vals = berger.columns[2:]
  berger = berger[berger[vals].fillna("").astype(str).apply(lambda r: "".join(r).strip() != "", axis=1)]
  save(berger, "berger_atlas_si_tables_raw", "сырые таблицы; заголовки столбцов — в первой строке каждой таблицы")
  save(pd.DataFrame(texts), "berger_atlas_si_page_text", "полный текст страниц — запасной вариант, если таблица разобралась плохо")
  berger.groupby("table_id").size()

## 13. C2DB (2D-материалы: края зон относительно вакуума)
Файл `c2db.db` выдаётся после регистрации на <https://cmr.fysik.dtu.dk/c2db/c2db.html>. Загрузите его (или положите в Drive и укажите путь).

In [18]:
RUN_C2DB = False
if RUN_C2DB:
    from ase.db import connect
    from google.colab import files
    C2DB_PATH = next(iter(files.upload()))     # или путь в Drive
    db = connect(C2DB_PATH)
    first = next(db.select(limit=1)); print("доступные ключи:", sorted(first.key_value_pairs)[:80])
    keys = ["uid", "gap", "gap_hse", "gap_gw", "vbm", "cbm", "vbm_hse", "cbm_hse", "evac", "work_function",
            "ehull", "hform", "dyn_stab", "magstate"]
    rows = []
    for r in db.select():
        els = set(r.symbols)
        if els & {"O", "S", "Se", "N", "C"} or els & set(HOSTS_METAL):
            d = {"formula": r.formula}; d.update({k: r.get(k) for k in keys})
            rows.append(d)
    c2 = pd.DataFrame(rows)
    save(c2, "c2db_selected", "края зон относительно вакуума: E_NHE ≈ −(E_vac_ref) − 4.44 эВ")

## 14. Корпуса CO₂RR (Sci. Data 2023 — ScienceDB; LLM-корпус Sci. Data 2024)
Скачайте файлы вручную со ScienceDB (ссылки: <https://www.scidb.cn/anonymous/NjdWeml5> — gold, <https://www.scidb.cn/anonymous/M3FNZjZi> — silver; для 2024 — ссылка в разделе Data Records статьи) и загрузите сюда.
Модуль читает csv/xlsx/json/jsonl, ищет строки, где упоминаются Au/Ag/Cu, и сохраняет их. Это корпуса извлечённых из текста сущностей — дальше нужна ручная/LLM-нормализация в таблицу «катализатор → продукт → FE».

In [19]:
RUN_CORPUS = False
if RUN_CORPUS:
    from google.colab import files
    up = files.upload()
    pat = re.compile(r"(?<![A-Za-z])(Au|Ag|Cu)(?![a-z])")
    for name, b in up.items():
        ext = name.lower().rsplit(".", 1)[-1]
        if ext == "csv": df = pd.read_csv(io.BytesIO(b))
        elif ext in ("xlsx", "xls"): df = pd.read_excel(io.BytesIO(b))
        elif ext == "jsonl": df = pd.read_json(io.BytesIO(b), lines=True)
        elif ext == "json":
            obj = json.loads(b); df = pd.json_normalize(obj if isinstance(obj, list) else obj.get("data", obj))
        else: print("пропуск:", name); continue
        print(name, df.shape, list(df.columns)[:15])
        txt = df.astype(str).agg(" ".join, axis=1)
        sub = df[txt.str.contains(pat)]
        save(sub, "corpus_" + re.sub(r"\W+", "_", name.rsplit(".", 1)[0])[:40], "строки с Au/Ag/Cu")

## 15. Слияние в `PIML_open_sources.sqlite` и Excel
Загрузите текущую базу (из чата) — все CSV из `/content/piml/out` добавятся в неё таблицами `mod_<имя>`. Готовые файлы скачаются автоматически.

In [20]:
from google.colab import files
up = files.upload()                       # PIML_open_sources.sqlite
db_name = next(iter(up)); db_path = Path("/content/piml") / db_name; db_path.write_bytes(up[db_name])
csvs = sorted(OUT.glob("*.csv"))
with sqlite3.connect(db_path) as con:
    for p in csvs:
        pd.read_csv(p).to_sql("mod_" + p.stem, con, if_exists="replace", index=False)
    log = pd.DataFrame({"table": ["mod_" + p.stem for p in csvs], "rows": [sum(1 for _ in open(p)) - 1 for p in csvs],
                        "built": time.strftime("%Y-%m-%d %H:%M")})
    log.to_sql("mod_build_log", con, if_exists="replace", index=False)
xlsx = Path("/content/piml/PIML_modules.xlsx")
with pd.ExcelWriter(xlsx) as xw:
    for p in csvs:
        df = pd.read_csv(p)
        if len(df) <= 1_000_000: df.to_excel(xw, sheet_name=p.stem[:31], index=False)
print(log.to_string(index=False))
if DRIVE_DIR:
    import shutil; shutil.copy(db_path, DRIVE_DIR); shutil.copy(xlsx, DRIVE_DIR)
files.download(str(db_path)); files.download(str(xlsx))

Saving PIML_open_sources.sqlite to PIML_open_sources.sqlite
                             table   rows            built
    mod_band_edges_all_gap_sources     51 2026-11-05 23:46
               mod_band_edges_best     20 2026-11-05 23:46
            mod_elements_mendeleev     36 2026-11-05 23:46
mod_jarvis_supports_all_polymorphs    315 2026-11-05 23:46
  mod_jarvis_supports_ground_state     20 2026-11-05 23:46
      mod_mok_dstar_by_composition     28 2026-11-05 23:46
        mod_mok_dstar_dft_surfaces    148 2026-11-05 23:46
    mod_mp_supports_all_polymorphs    868 2026-11-05 23:46
      mod_mp_supports_ground_state     20 2026-11-05 23:46
  mod_ocx24_processed_co2rr_AuAgCu    147 2026-11-05 23:46
       mod_sac_MC4xNx_raw_energies    140 2026-11-05 23:46
                 mod_tmqm_by_metal     21 2026-11-05 23:46
   mod_tmqm_single_metal_complexes 100396 2026-11-05 23:46


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>